# 01 — Collection

Runs the module's collectors in order and reports what each produced. Every step
is **resumable and idempotent**: one cache file per unit of work, and a unit
already on disk is never re-requested. Re-running this notebook costs no quota
for anything already collected.

Order matters. The free, unmetered sources run first, then the quota-limited
ones, then the paid one.

| step | source | cost | what it gives |
|---|---|---|---|
| 0 | probe | free | what each provider actually delivers |
| 1 | Finnhub calendar | free | Tier A census — the denominator |
| 2 | EDGAR submissions | free | Tier B watchlist: CIK, S-1, listing |
| 3 | EDGAR submissions | **free, no network** | DRS / Form D / comment letters / withdrawals |
| 4 | Wikipedia pageviews | **free, keyless** | dense monthly attention, 2015→today |
| 5 | NYT Article Search | 500/day | sparse elite-press attention |
| 6 | Polygon | 5/min | daily price, rolling 2-year entitlement |
| 7 | twitterapis.com | **paid per call** | X density — *account credits exhausted* |

**Read `../README.md` first.** Three of the seven sources in the original module
brief did not survive measurement, and the reasons shape everything below.

In [ ]:
import subprocess, sys, json
import pandas as pd
sys.path.insert(0, "../..")          # repo root, so `research.collect` imports

from research.collect import paths

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

def run(*args):
    """Run a collector as a subprocess and show its tail.

    A subprocess rather than an in-process import: the collectors are the
    supported entry point and are what a shell re-run would execute, so the
    notebook should not exercise a different path than the README documents.
    """
    cmd = [sys.executable, "-m", *args]
    print("$", " ".join(cmd[2:]))
    p = subprocess.run(cmd, cwd="../..", capture_output=True, text=True)
    tail = [l for l in p.stderr.splitlines() if "HTTP Request" not in l]
    print("\n".join(tail[-14:]))
    if p.stdout.strip():
        print(p.stdout.strip())
    return p.returncode

## Step 0 — Measure the sources

Not optional preamble. Free-tier entitlements change, and every downstream design
choice follows from what these calls return. Output lands in
`data/source_probe.json` with a timestamp.

Takes a few minutes: NYT is 5 requests/minute and the Polygon entitlement check
is a bisection.

In [ ]:
run("research.collect.probe_sources")

In [ ]:
probe = json.loads(paths.SOURCE_PROBE_JSON.read_text())
print("measured", probe["probed_at"], "\n")
for p in probe["probes"]:
    print(f"[{p['verdict'].upper():7}] {p['source']:12} {p['question']}")
    print(f"          {p['detail'][:150]}")

## Step 1 — Tier A census

The complete Finnhub IPO calendar, 2019-01-01 to today, one request per month
window. Month-at-a-time because the endpoint returns a truncated array for
multi-year ranges without saying so — the failure mode that silently shortens a
census.

Raw JSON per window is written first; normalization reads only those files, so
`--normalize-only` rebuilds the census with the network off.

In [ ]:
run("research.collect.finnhub_census")

In [ ]:
census = pd.read_parquet(paths.CENSUS_PARQUET)
census["year"] = pd.to_datetime(census["calendar_date"]).dt.year
print(f"{len(census)} rows, {census['calendar_date'].min()} to {census['calendar_date'].max()}\n")
display(pd.crosstab(census["year"], census["status"], margins=True))

dropped = pd.read_csv(paths.CENSUS_DROPPED_CSV)
print(f"\ndropped {len(dropped)} rows:")
print(dropped["drop_reason"].value_counts().to_string() if len(dropped) else "  none")

### Is the census actually complete?

A silently truncated count looks exactly like a correct one, so it gets checked
against an independent number. Splitting the priced rows with a crude SPAC
heuristic should reproduce the widely-reported 2021 split of roughly 610 SPAC
IPOs and 400 traditional ones.

The heuristic is used **only** for this check and nowhere in the analysis.

In [ ]:
priced = census[census["status"] == "priced"].copy()
sym = priced["symbol"].fillna("")
priced["spac_like"] = (sym.str.endswith("U") | sym.str.contains("'U")
                       | priced["name"].str.contains("Acquisition", case=False, na=False))
tab = pd.crosstab(priced["year"], priced["spac_like"]).rename(
    columns={False: "operating", True: "SPAC-like"})
display(tab)
print("2021 reported externally as ~610 SPAC / ~400 traditional; "
      f"this census gives {tab.loc[2021, 'SPAC-like']} / {tab.loc[2021, 'operating']}.")

## Step 2 — Tier B watchlist

Enriches the committed seed CSV against EDGAR and the Tier A census. The seed's
`Year`/`Month` columns are month-granularity and unsourced, so nothing here
trusts them: every row is resolved independently and `date_flag` records
agreement or disagreement rather than overwriting.

`--price-floor` is passed explicitly rather than derived from today, so a re-run
reproduces an earlier watchlist instead of reclassifying rows as the price
feed's rolling window slides.

In [ ]:
run("research.collect.edgar_enrich")

In [ ]:
from research.collect.edgar_enrich import read_watchlist_df
wl = read_watchlist_df()          # NOT pd.read_csv: it turns the CIK into a float
print(f"{len(wl)} rows, {int(wl['tier_b'].sum())} in Tier B\n")
print("by collection priority:")
print(wl["tier_b_priority"].value_counts().sort_index().to_string(), "\n")
print("date cross-check against the seed:")
print(wl["date_flag"].str.slice(0, 46).value_counts().to_string(), "\n")
print("excluded rows and why:")
display(wl.loc[~wl["tier_b"], ["company", "method", "seed_exchange", "exclude_reason"]])

## Step 3 — The pre-listing event timeline (free, no network)

This step adds **no new provider**. Every filing it reads was already downloaded
in Step 2, which used it for two dates and discarded the rest. The rest turns
out to be the most informative material in the module.

**DRS** is the confidential draft registration statement. Under the JOBS Act an
emerging growth company files one before its public S-1, and EDGAR exposes it
only once the company actually goes public. It is therefore a real event that was
*secret at the time* — which makes it a much better treatment date than the S-1.

Also extracted: **Form D** (private placements — a pre-IPO funding timeline),
**CORRESP/UPLOAD** (SEC comment letters, i.e. registration friction), **RW**
(actual withdrawals), and amendment counts.

In [ ]:
run("research.collect.edgar_events", "--offline")

In [ ]:
from research.collect.edgar_events import EVENTS_PARQUET
ev = pd.read_parquet(EVENTS_PARQUET)
gaps = ev["drs_to_s1_days"].dropna()
print(f"DRS present for {ev['drs_first'].notna().sum()} of {len(ev)} companies")
print(f"DRS -> public S-1 gap: median {gaps.median():.0f} d, "
      f"quartiles {gaps.quantile(.25):.0f}/{gaps.quantile(.75):.0f}, max {gaps.max():.0f}\n")
display(ev.sort_values("drs_to_s1_days", ascending=False)[
    ["company", "drs_first", "s1_first", "drs_to_s1_days",
     "s1_amendments", "form_d_count", "comment_rounds"]].head(12))

The long tail matters for study design. A company whose DRS precedes its
S-1 by 1,408 days had been in registration for nearly four years before the
public learned — so a 24-month pre-S-1 window does not even reach its
confidential filing, and "attention before the S-1" quietly includes years of
already-being-in-registration.

In [ ]:
print("companies whose DRS falls outside a 24-month pre-S-1 window:")
outside = ev[ev["drs_to_s1_days"] > 730][["company", "drs_first", "s1_first",
                                          "drs_to_s1_days"]]
display(outside)
print("\nregistration friction — SEC comment rounds and S-1 amendments:")
display(ev[["company", "comment_rounds", "s1_amendments", "drs_amendments"]]
        .sort_values("comment_rounds", ascending=False).head(8))

## Step 4 — Wikipedia pageviews (free, keyless)

The dense attention series, and the one that fixes the sample-size problem: no
API key, no per-call cost, no meaningful rate limit, monthly granularity in one
request per article, coverage from **2015-07-01** to today.

This is **not** Google Trends. `docs/google-trends-decision.md` rejected Trends
for being a *normalized* index (relative to its own query and window) and
*sampled* (the same query twice gives different numbers). Pageviews are absolute
counts and deterministic. They do share Trends' third property — an aggregate,
not discrete events — which is why this lives in `research/` and is never offered
to the deployed `SourceAdapter`.

Three data-integrity problems had to be solved, and each is visible in the
output below rather than assumed away.

In [ ]:
run("research.collect.wikipedia", "--cohort", "tier_b")

### Problem 1 — entity resolution

Search resolves a brand to the wrong subject often enough that the mapping has to
be audited, not trusted. Every resolution and its candidate list is logged.

In [ ]:
res = pd.read_csv(paths.DATA / "wikipedia_resolution.csv")
print(res["method"].value_counts().to_string(), "\n")
display(res.loc[res["method"] == "curated override",
                ["company", "title", "notes"]].head(20))

Four titles were wrong on the first pass and were caught by this log:
Robinhood resolved to a *Robin Hood* disambiguation page, Peloton to the cycling
term, Palantir needed checking against the Tolkien object, and Medline to the NIH
bibliographic database. Two more were *correctly named but nearly unvisited* —
`Reddit, Inc.` gets 3 views/month against 210,000 for `Reddit`, and
`Chime Financial` has no traffic at all against 11,000 for `Chime (company)`.

### Problem 2 — pageviews are keyed on the title string, not the article

So a renamed page leaves its history behind. The collector sums the article
*and its redirects*, which both repairs the truncation and measures the thing we
actually want: attention to the entity, however the reader navigated to it.

In [ ]:
blob = json.loads((paths.RAW / "wikipedia" / "zoom.json").read_text())
print("Zoom — views by contributing title:")
for title, views in sorted(blob["views_by_title"].items(), key=lambda kv: -kv[1]):
    print(f"  {views:>10,}  {title}")
print("\nThe current title holds a small fraction of the total. Without the "
      "redirect merge Zoom's median month reads 16 views instead of ~23,000.")

### Problem 3 — absent months are not zero months

The API **omits** months before the article existed rather than returning zero.
Zero-filling them would be wrong, and worse, it would manufacture exactly the
"attention rises before the filing" shape this module is testing for — an
article created at IPO time makes every earlier month look quiet.

A second hazard is subtler: a title can be *repurposed*. The `Bullish` article
dates from 2005, when the title held the market term, and was rewritten for the
2021 crypto exchange. Creation date cannot see that. So each company also gets a
floor from its own earliest EDGAR trace (Form D, DRS or S-1), and only
`views_valid` is gated — raw `views` is always retained.

In [ ]:
from research.collect.wikipedia import WIKI_PARQUET
wiki = pd.read_parquet(WIKI_PARQUET)
print(f"{len(wiki)} company-months; {int(wiki['valid_attention'].sum())} valid, "
      f"{int((~wiki['valid_attention']).sum())} excluded as pre-article or pre-EDGAR\n")
g = wiki.groupby("company").agg(months=("views", "size"),
                                valid=("valid_attention", "sum"),
                                peak_raw=("views", "max"),
                                peak_valid=("views_valid", "max"),
                                floor=("floor_source", "first"))
print("where the floor changed the answer most:")
g["clipped"] = g["peak_raw"] - g["peak_valid"].fillna(0)
display(g.sort_values("clipped", ascending=False).head(6))

Circle is the clearest case: its raw peak of 45,790 falls to 8,066 once the
floor applies, because the earlier traffic belonged to a previous occupant of the
`Circle (company)` title.

One hand-verified override exists, and it is deliberately the only kind of
exception: Klarna's article legitimately dates to 2010 but its first US EDGAR
filing is 2023 (it is a foreign private issuer), so the automatic floor would
discard thirteen years of genuine attention. That case and a repurposed title
look identical to any heuristic, so the handful that matter are checked by hand
and recorded in `HAND_VERIFIED_FLOOR`.

## Step 5 — NYT monthly counts

The primary *news* signal, and the only news source that survived measurement:
GNews grants 30 days of history on the free plan and cannot see a pre-S-1 window
at all.

Limits are ~5/minute and ~500/day. The minute limit is request spacing; the daily
limit is a hard stop the collector enforces itself, because the full watchlist
exceeds one day's quota. `--priority 1` collects the 12 companies whose listings
the price feed also covers — 374 company-months, which fits one day.

Roughly 75 minutes at 12s spacing, and resumable: stop it and re-run and it
continues from the cache.

In [ ]:
run("research.collect.nyt", "--priority", "1", "--budget", "480")

In [ ]:
nyt = pd.read_parquet(paths.NYT_COUNTS_PARQUET)
print(f"{len(nyt)} company-months, {nyt['company'].nunique()} companies\n")
print("count distribution — note how much of it is zero:")
print(nyt["count"].describe().to_string())
print(f"\nzero-count months: {(nyt['count'] == 0).sum()} of {len(nyt)}"
      f" ({100 * (nyt['count'] == 0).mean():.0f}%)")
print("\nThis is a thin instrument. Instacart's peak IPO month is 17 articles.")

### Hand-check the matches

The counts are only as good as the query form, so check a sample by eye. The
first page of article metadata came back in the same response as each count, so
this costs no quota.

Look for: does the headline actually concern the company, or did an
ordinary-word brand name pull in something unrelated?

In [ ]:
import random
random.seed(0)   # fixed, so the sample is the same on every re-run

files = [p for c in paths.RAW_NYT.iterdir() if c.is_dir() for p in c.glob("*.json")]
for path in sorted(random.sample(files, min(8, len(files)))):
    blob = json.loads(path.read_text())
    print(f"\n=== {blob['company']} {blob['month']}  q={blob['query']}  hits={blob['hits']}")
    for doc in (blob.get("docs") or [])[:3]:
        print("   ", (doc.get("headline") or {}).get("main", "")[:96])

## Step 6 — Price series

Polygon, the licensed feed the deployed pipeline already uses.
Finnhub's candle endpoint returns 403 on this key, and no scraped source was
substituted for it — see the README.

The plan grants a **rolling two-year** window, so this covers only the 12
priority-1 rows. It also reconciles the calendar listing date against the first
traded session; where they disagree, the tape wins.

In [ ]:
run("research.collect.prices", "--priority", "1")

In [ ]:
recon = pd.read_csv(paths.DATA / "price_reconciliation.csv")
display(recon[["company", "ticker", "calendar_date", "first_trade_date", "verdict"]])
px = pd.read_parquet(paths.PRICES_PARQUET)
print(f"\n{len(px)} bars across {px['company'].nunique()} companies")
print("\nsessions available (90 needed for a full-window return):")
print(px.groupby("company")["session"].max().add(1).sort_values().to_string())

## Step 7 — X / Twitter density (paid; currently blocked)

**The account's prepaid balance is exhausted** (`HTTP 402 insufficient_credits`),
so there is no X data and `twitter_monthly_counts.parquet` does not exist. The
collector is finished and correct; it needs a topped-up balance.

Two safeguards now exist because of how the balance went:

- **`--recompute`** rebuilds every month's metrics from the stored timestamps
  with no network call and no cost. Re-collecting to fix an arithmetic bug spends
  money to obtain bytes already on disk.
- **A month whose requests errored is never cached.** The cache is also the
  resume mechanism, so writing a failed month as "0 tweets" would make a later
  run skip it and read an API failure as a measured zero. A 402 aborts the whole
  run and writes nothing.

The cell below demonstrates the abort path. It costs nothing.

In [ ]:
run("research.collect.twitter", "--calibrate", "--priority", "1",
    "--budget", "30", "--max-pages", "6", "--anchors", "3")
print("\nNothing cached, as intended:",
      len(list((paths.RAW / "twitter").glob("*/*.json"))), "files")

### Sizing a run, if the balance is topped up

Cost is fixed per company-month, so the arithmetic is simple. Decide the spend
before running.

In [ ]:
from research.collect.twitter import COST_PER_CALL_USD
from research.collect.nyt import load_tier_b, month_starts, window_for

for priority in (1, 9):
    rows = load_tier_b(priority)
    months = sum(len(month_starts(*w)) for r in rows
                 if (w := window_for(r)) is not None)
    for pages in (6, 10, 20):
        print(f"priority<={priority}: {len(rows):2d} companies, {months:4d} "
              f"company-months, {pages:2d} pages each -> {months * pages:5d} calls "
              f"= ${months * pages * COST_PER_CALL_USD:6.2f}")
    print()

## Provenance

Everything above is reproducible from the cached payloads with the network off:

```bash
python -m research.collect.finnhub_census --normalize-only
python -m research.collect.edgar_events   --offline
python -m research.collect.wikipedia      --aggregate-only
python -m research.collect.nyt            --aggregate-only
python -m research.collect.prices         --aggregate-only
python -m research.collect.twitter        --recompute      # free; no network
```

In [ ]:
for path in sorted(paths.DATA.rglob("*")):
    if path.is_file() and path.suffix in {".parquet", ".csv", ".json"} \
            and "raw" not in path.parts:
        print(f"{str(path.relative_to(paths.DATA)):46} {path.stat().st_size / 1024:9.1f} KB")
print(f"\nraw payloads cached: {sum(1 for _ in paths.RAW.rglob('*.json'))} files")